Audit the raw data:
Before cleaning the data, you must inspect it and report what is wrong.

This process is called a data audit.

In [1]:
import pandas as pd

orders = pd.read_csv("data/raw/orders.csv")
customers = pd.read_csv("data/raw/customers.csv")
products = pd.read_csv("data/raw/products.csv")

print(orders.shape)
print(orders.dtypes)

print("Order Table ")
print(orders.head(5))

(13084, 9)
order_id                str
customer_id             str
product_id              str
quantity              int64
order_ts                str
status                  str
discount_pct          int64
order_amount            str
delivery_minutes    float64
dtype: object
Order Table 
     order_id customer_id product_id  quantity             order_ts  \
0  ORD-108250       C0792       P004         2  2025-09-01 12:24:59   
1  ORD-101939       C0190       P003         1  2025-08-17 07:07:23   
2  ORD-108765       C0841       P033         1  2025-07-13 09:42:32   
3  ORD-108861       C0852       P010         1  2025-07-04 11:23:24   
4  ORD-109150       C0877       P018         2  2025-05-07 15:35:01   

      status  discount_pct order_amount  delivery_minutes  
0  completed             0        310.0              17.0  
1  completed             5       232.75              21.0  
2  completed            10       1349.1              27.0  
3  completed             0         45.0    

To count the missing values

In [2]:
print(orders.isnull().sum())

order_id               0
customer_id            0
product_id             0
quantity               0
order_ts               0
status                 0
discount_pct           0
order_amount          53
delivery_minutes    2541
dtype: int64


For tickets.csv, if you use it later, you must load it with:

In [3]:
tickets = pd.read_csv(
    "data/raw/tickets.csv",
    keep_default_na=False
)

This ensures empty ticket texts remain empty strings rather than automatically becoming missing values.

Use the actual supplied file paths if your project uses different ones.

Identify duplicates and invalid values

In [4]:
# Exact duplicate rows
print(orders.duplicated().sum())

# Pending orders
print((orders["status"] == "pending").sum())

# Distinct order IDs
print(orders["order_id"].nunique())

# Amounts that cannot be converted to numbers
amounts = pd.to_numeric(
    orders["order_amount"], errors="coerce"
)
print((amounts.isna() & orders["order_amount"].notna()).sum())

119
656
12309
67


Clean the orders table 
This is the most important cleaning task because it carries the most marks in Part A.

You must follow the given order of operations.

Step 1: Convert the order timestamp

In [5]:
orders["order_ts"] = pd.to_datetime(
    orders["order_ts"], errors="coerce"
)

A timestamp represents when the order was recorded.

Step 2: Remove exact duplicates

In [6]:
orders = orders.drop_duplicates()

An exact duplicate means every column in one row matches the corresponding column in another row.

Step 3: Keep the latest record for each order
Consider this example:
order_id	order_ts	status
O101	10:00	pending
O101	10:30	completed

These rows describe the same order. The later record is the final record, so keep the completed row.

One possible implementation is: 

In [7]:
orders = orders.sort_values("order_ts")

orders = orders.drop_duplicates(
    subset="order_id",
    keep="last"
)

Sort chronologically first, then keep the last row.

Step 4: Fix order amounts
Some order amounts contain invalid text or are missing.

Convert them to numbers:

In [8]:
# Amounts that are not numeric converted to numeric
orders['order_amount'] = pd.to_numeric(
    orders["order_amount"], errors="coerce"
)

# Fill missing values with their respective means
orders["order_amount"] = orders["order_amount"].fillna(
    orders["order_amount"].mean()
)

Step 5: Fix delivery times
Negative delivery times are invalid, so replace them with missing values.

In [9]:
orders.loc[
    orders["delivery_minutes"] < 0,
    "delivery_minutes"
] = float("nan")

In [10]:
# Convert columns to numeric
orders["delivery_minutes"] = pd.to_numeric(
    orders["delivery_minutes"], errors="coerce"
)

# Fill missing values with their respective means
orders["delivery_minutes"] = orders["delivery_minutes"].fillna(
    orders["delivery_minutes"].mean()
)

Now handle the missing values differently according to order status.

For completed orders: Fill missing delivery times with the median delivery time for that product category, calculated from completed orders.

For cancelled orders: Keep delivery times missing. A cancelled order was never delivered, so inventing a delivery time would be incorrect.

Conceptually, the code looks like this:

Step 1: Check the columns in orders

In [11]:
print(orders.columns.tolist())

['order_id', 'customer_id', 'product_id', 'quantity', 'order_ts', 'status', 'discount_pct', 'order_amount', 'delivery_minutes']


Now that category is missing

Step 2: Why is category missing?
In your Instamart assignment, the product category may be stored in the separate products DataFrame rather than in orders.

For example, the products table might look like this:
product_id	product_name	category
P001	Rice	Grocery
P002	Headphones	Electronics
P003	Milk	Dairy

Your orders table contains product_id, which you can use to connect the two tables.

Step 3: Add the product category to orders
First, check that products has these columns:

In [12]:
print(products.columns.tolist())

['product_id', 'product_name', 'category', 'price_inr']


If both DataFrames have product_id and products contains category, merge them:

In [13]:
orders = orders.merge(
    products[["product_id", "category"]],
    on="product_id",
    how="left"
)

This adds the appropriate category to each order.

Important: If you already merged the category or are unsure about your column names, check the output of Step 1 first. Avoid merging the same category column multiple times.

Step 4: Run your original code again
Once the category column is available, run:

In [14]:
completed = orders["status"].eq("completed")

category_median = (
    orders.loc[completed]
    .groupby("category")["delivery_minutes"]
    .transform("median")
)

orders.loc[completed, "delivery_minutes"] = (
    orders.loc[completed, "delivery_minutes"]
    .fillna(category_median)
)

What does this code do?
1. Identifies completed orders.
2. Calculates the median delivery time for each product category among completed orders.
3. Fills missing delivery times for completed orders using the corresponding category median.
4. Leaves cancelled orders' delivery times missing.
This follows the cleaning rule in your assignment.

Step 5: Validate the result

In [15]:
print("Category column exists:", "category" in orders.columns)

print(
    "Missing delivery times in completed orders:",
    orders.loc[
        orders["status"].eq("completed"),
        "delivery_minutes"
    ].isna().sum()
)

print(
    "Missing delivery times in cancelled orders:",
    orders.loc[
        orders["status"].eq("cancelled"),
        "delivery_minutes"
    ].isna().sum()
)

Category column exists: True
Missing delivery times in completed orders: 0
Missing delivery times in cancelled orders: 0


In [16]:
# Exact duplicate rows
print("Duplicated rows count:")
print(orders.duplicated().sum())

print(orders.isnull().sum())
print(orders.head(25))


Duplicated rows count:
0
order_id            0
customer_id         0
product_id          0
quantity            0
order_ts            0
status              0
discount_pct        0
order_amount        0
delivery_minutes    0
category            0
dtype: int64
      order_id customer_id product_id  quantity            order_ts  \
0   ORD-110093       C0967       P020         1 2025-04-01 07:08:28   
1   ORD-111396       C1117       P036         2 2025-04-01 07:11:59   
2   ORD-102006       C0194       P009         1 2025-04-01 07:12:43   
3   ORD-105921       C0569       P005         2 2025-04-01 07:54:43   
4   ORD-108115       C0776       P012         2 2025-04-01 08:02:58   
5   ORD-111469       C1122       P017         3 2025-04-01 08:26:52   
6   ORD-102351       C0224       P039         1 2025-04-01 08:30:09   
7   ORD-107027       C0674       P006         3 2025-04-01 08:40:05   
8   ORD-103459       C0326       P011         2 2025-04-01 08:49:20   
9   ORD-100900       C0084      

Now we are going to clean the customer table. 

In [17]:
print("Customer table data types:")
print(customers.dtypes)

print("No of Rows and Columns:")
print(customers.shape)

print("Null values count:")
print(customers.isnull().sum())

print("Duplicated values count:")
print(customers.duplicated().sum())

Customer table data types:
customer_id        str
city               str
signup_date        str
age            float64
gender             str
dtype: object
No of Rows and Columns:
(1200, 5)
Null values count:
customer_id     0
city            0
signup_date     0
age            61
gender         30
dtype: int64
Duplicated values count:
0


In [18]:
print(customers.head(5))

  customer_id       city signup_date   age  gender
0       C0001    Chennai  2024-01-13  35.0    MALE
1       C0002      DELHI  2024-02-17  31.0    MALE
2       C0003  bengaluru  2025-02-18  18.0       F
3       C0004  Bengaluru  2024-06-21  24.0  FEMALE
4       C0005  Bengaluru  2024-10-01  25.0  FEMALE


First we have to clean the city column. It contains not all the city in the same pattern like full capse, first letter capse remaining small. So we have to change it in same pattern.

In [19]:
clean_map = {
    'CHENNAI' : 'Chennai',
    'chennai' : 'Chennai',
    'BENGALURU' : 'Bengaluru',
    'bengaluru' : 'Bengaluru',
    'HYDERABAD' : 'Hyderabad',
    'hyderabad' : 'Hyderabad',
    'DELHI' : 'Delhi',
    'delhi' : 'Delhi',
    'MUMBAI': 'Mumbai',
    'mumbai': 'Mumbai',
    'PUNE' : 'Pune',
    'pune' : 'Pune'
}

customers['city'] = customers['city'].replace(clean_map)
print(customers.head(5))

  customer_id       city signup_date   age  gender
0       C0001    Chennai  2024-01-13  35.0    MALE
1       C0002      Delhi  2024-02-17  31.0    MALE
2       C0003  Bengaluru  2025-02-18  18.0       F
3       C0004  Bengaluru  2024-06-21  24.0  FEMALE
4       C0005  Bengaluru  2024-10-01  25.0  FEMALE


Correct missing or invalid ages
The rules say:
- Ages below 10 or above 100 are invalid.
- Invalid ages must become missing.
- All missing ages must then be filled with the median age of the customer's city.

The median is 30.

A Pune customer with a missing age receives 30, rather than the median age of all customers across every city.

Use groupby() and transform() as requested:

In [20]:
customers.loc[
    ~customers["age"].between(10, 100),
    "age"
] = float("nan")

customers["age"] = customers["age"].fillna(
    customers.groupby("city")["age"].transform("median")
)

print(customers.isnull().sum())


customer_id     0
city            0
signup_date     0
age             0
gender         30
dtype: int64


Convert signup dates

In [21]:
customers["signup_date"] = pd.to_datetime(
    customers["signup_date"], errors="coerce"
)

In [22]:
clean_map = {
    'MALE' : 'Male',
    'male' : 'Male',
    'M' : 'Male',
    'm' : 'Male',
    'FEMALE' : 'Female',
    'female' : 'Female',
    'F' : 'Female',
    'f' : 'Female'
}

customers['gender'] = customers['gender'].replace(clean_map)
print(customers.head(5))

  customer_id       city signup_date   age  gender
0       C0001    Chennai  2024-01-13  35.0    Male
1       C0002      Delhi  2024-02-17  31.0    Male
2       C0003  Bengaluru  2025-02-18  18.0  Female
3       C0004  Bengaluru  2024-06-21  24.0  Female
4       C0005  Bengaluru  2024-10-01  25.0  Female


In [23]:
# Calculate the mode of gender
mode = customers['gender'].mode()[0]

# Fill missing gender with mode
customers['gender'] = customers['gender'].fillna(mode)

print(customers.isnull().sum())
print(customers.head(65))

customer_id    0
city           0
signup_date    0
age            0
gender         0
dtype: int64
   customer_id       city signup_date   age  gender
0        C0001    Chennai  2024-01-13  35.0    Male
1        C0002      Delhi  2024-02-17  31.0    Male
2        C0003  Bengaluru  2025-02-18  18.0  Female
3        C0004  Bengaluru  2024-06-21  24.0  Female
4        C0005  Bengaluru  2024-10-01  25.0  Female
..         ...        ...         ...   ...     ...
60       C0061     Mumbai  2025-03-03  44.0    Male
61       C0062     Mumbai  2024-07-18  39.0    Male
62       C0063       Pune  2024-07-20  55.0  Female
63       C0064  Bengaluru  2024-04-03  21.0  Female
64       C0065    Chennai  2025-01-20  44.0    Male

[65 rows x 5 columns]


Next we have to clean the products table duplicates and null value checks.

In [21]:
print("Products table data types:")
print(products.dtypes)

print("No of Rows and Columns:")
print(products.shape)

print("Null values count:")
print(products.isnull().sum())

print("Duplicated values count:")
print(products.duplicated().sum())

Products table data types:
product_id        str
product_name      str
category          str
price_inr       int64
dtype: object
No of Rows and Columns:
(40, 4)
Null values count:
product_id      0
product_name    0
category        0
price_inr       0
dtype: int64
Duplicated values count:
0


The products table is already clean. It does not contain missing values or duplicates. 

After cleaning, you must save the results in two formats.
Step 1: Save clean CSV files

In [24]:
customers.to_csv(
    "data/clean/customers.csv", index=False
)

products.to_csv(
    "data/clean/products.csv", index=False
)

orders.to_csv(
    "data/clean/orders.csv", index=False
)

Step 2: Save the data in SQLite
SQLite is a small database that stores tables inside a database file.

If you want to connect to the SQLite database
Use Python's built-in sqlite3 module:

In [2]:
import sqlite3

# Connect to the database
conn = sqlite3.connect("data/instamart.db")

print("Database connected successfully!")

Database connected successfully!


If you want to check whether the database file exists

In [3]:
import os

path = "data/instamart.db"

if os.path.exists(path):
    print("Database file exists")
else:
    print("Database file not found")

Database file exists


It must contain three tables:
- customers
- products
- orders

In [4]:
connection = sqlite3.connect("data/instamart.db")

customers.to_sql(
    "customers", connection,
    if_exists="replace", index=False
)

products.to_sql(
    "products", connection,
    if_exists="replace", index=False
)

orders.to_sql(
    "orders", connection,
    if_exists="replace", index=False
)

connection.close()

Before saving, format order_ts as text in the specified format, YYYY-MM-DD HH:MM:SS.

Step 3: Create a validation script
A validation script checks that your cleaned data matches the required results.

In [28]:
print("Total rows:", len(orders))
print("Unique order IDs:", orders["order_id"].nunique())
print("Missing order amounts:", orders["order_amount"].isna().sum())
print("Pending orders:", (orders["status"] == "pending").sum())

Total rows: 12309
Unique order IDs: 12309
Missing order amounts: 0
Pending orders: 0


In [29]:
assert len(orders) == 12309
assert orders["order_id"].nunique() == 12309
assert orders["order_amount"].isna().sum() == 0
assert (orders["status"] == "pending").sum() == 0

In simple words: Save the cleaned data in files and a database, then prove it meets the requirements.

Task 6: SQL analysis

Now that the data is clean, use SQL to answer seven business questions.

You must save the queries in sql/queries.sql, execute each from Python using pd.read_sql_query(), and report the results.

SQL is a language used to retrieve and analyse information in databases.

pandas cross-check and exploratory data analysis 
You will use Python and pandas to verify your SQL calculations and understand the patterns in the data.

The following examples assume you have a DataFrame named orders containing the cleaned data, and that its status column identifies completed orders.

In [5]:
import pandas as pd
import numpy as np

# Check the columns in both DataFrames
print("Orders columns:", orders.columns.tolist())
print("Customers columns:", customers.columns.tolist())

# Merge city from customers into orders
orders = pd.merge(
    orders,
    customers[["customer_id", "city"]],
    on="customer_id",
    how="left"
)

Orders columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'order_ts', 'status', 'discount_pct', 'order_amount', 'delivery_minutes']
Customers columns: ['customer_id', 'city', 'signup_date', 'age', 'gender']


STEP 1: Check the column names

In [6]:
print("Available columns:", orders.columns.tolist())

required_columns = [
    "city", "status", "order_amount",
    "discount_pct", "quantity", "delivery_minutes"
]

missing_columns = [col for col in required_columns if col not in orders.columns]

if missing_columns:
    raise KeyError(
        f"Missing columns: {missing_columns}. "
        "Check the column names or merge the city column from the customers data."
    )

Available columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'order_ts', 'status', 'discount_pct', 'order_amount', 'delivery_minutes', 'city']


STEP 2: Clean columns and select completed orders

In [7]:
orders["status"] = orders["status"].astype("string").str.strip().str.lower()

orders["order_amount"] = pd.to_numeric(
    orders["order_amount"], errors="coerce"
)

for col in ["discount_pct", "quantity", "delivery_minutes"]:
    orders[col] = pd.to_numeric(orders[col], errors="coerce")

completed_orders = orders.loc[
    orders["status"].eq("completed")
].copy()

print("\nNumber of completed orders:", len(completed_orders))


Number of completed orders: 11217


PIVOT TABLE WITH GRAND TOTAL

In [8]:
# Sum completed order_amount for each city.
# margins=True adds an "All" row with the grand total.

city_pivot = pd.pivot_table(
    completed_orders,
    values="order_amount",
    index="city",
    aggfunc="sum",
    margins=True,
    margins_name="All",
    observed=True
)

print("\n1. City-wise pivot table with grand total:")
print(city_pivot)

pivot_total = city_pivot.loc["All", "order_amount"]

print(f"\nPivot table grand total: ₹{pivot_total:,.2f}")


1. City-wise pivot table with grand total:
             order_amount
city                     
 Bengaluru       55941.45
 Chennai         46498.60
 Delhi           87060.65
 Hyderabad       54081.90
 Mumbai          46375.20
 Pune            47703.25
BENGALURU        89130.95
Bengaluru       288005.40
CHENNAI          41265.60
Chennai         268884.05
DELHI            55279.10
Delhi           281820.95
HYDERABAD        31148.50
Hyderabad       345684.95
MUMBAI           40427.75
Mumbai          325691.15
PUNE             41846.80
Pune            335670.95
bengaluru        88448.55
chennai         105178.60
delhi            81399.65
hyderabad        98834.60
mumbai           64703.75
pune             75198.05
All            2996280.40

Pivot table grand total: ₹2,996,280.40


Compare the pivot total with the SQL total

In [9]:
# This assumes you already have a SQLite connection
# named conn and a database table named orders.
import sqlite3
import pandas as pd

# Connect to your SQLite database
conn = sqlite3.connect("data/instamart.db")

sql_result = pd.read_sql_query(
    """
    SELECT SUM(order_amount) AS sql_total
    FROM orders
    WHERE LOWER(TRIM(status)) = 'completed'
    """,
    conn
)

sql_total = sql_result.loc[0, "sql_total"]

print(f"SQL completed-order total: ₹{sql_total:,.2f}")

print("Do the totals match?",
      np.isclose(pivot_total, sql_total, rtol=1e-9, atol=0.01))

SQL completed-order total: ₹2,996,280.40
Do the totals match? True


Compare the pivot total with the SQL total

In [10]:
# This assumes you already have a SQLite connection
# named conn and a database table named orders.

sql_result = pd.read_sql_query(
    """
    SELECT SUM(order_amount) AS sql_total
    FROM orders
    WHERE LOWER(TRIM(status)) = 'completed'
    """,
    conn
)

sql_total = sql_result.loc[0, "sql_total"]

print(f"SQL completed-order total: ₹{sql_total:,.2f}")

print("Do the totals match?",
      np.isclose(pivot_total, sql_total, rtol=1e-9, atol=0.01))

SQL completed-order total: ₹2,996,280.40
Do the totals match? True


CITY-WISE SUM, MEAN AND COUNT

In [16]:
city_summary = (
    completed_orders
    .groupby("city")
    .agg(
        revenue_sum=("order_amount", "sum"),
        mean_order_amount=("order_amount", "mean"),
        order_count=("order_amount", "count")
    )
    .reset_index()
)

print("\n2. City-wise completed-order summary:")
print(city_summary.round(2).to_string(index=False))

# Check the reference cities
print("\nReference check for Bengaluru and Mumbai:")
print(
    city_summary[
        city_summary["city"].astype("string").str.strip().str.lower()
        .isin(["bengaluru", "mumbai"])
    ].round(2).to_string(index=False)
)



2. City-wise completed-order summary:
       city  revenue_sum  mean_order_amount  order_count
 Bengaluru      55941.45             274.22          204
   Chennai      46498.60             220.37          211
     Delhi      87060.65             268.71          324
 Hyderabad      54081.90             378.20          143
    Mumbai      46375.20             227.33          204
      Pune      47703.25             268.00          178
  BENGALURU     89130.95             276.80          322
  Bengaluru    288005.40             269.67         1068
    CHENNAI     41265.60             238.53          173
    Chennai    268884.05             278.64          965
      DELHI     55279.10             252.42          219
      Delhi    281820.95             245.27         1149
  HYDERABAD     31148.50             283.17          110
  Hyderabad    345684.95             286.40         1207
     MUMBAI     40427.75             284.70          142
     Mumbai    325691.15             337.50      

DESCRIPTIVE STATISTICS

In [11]:
amounts = completed_orders["order_amount"].dropna()

mean_amount = amounts.mean()
median_amount = amounts.median()
skewness = amounts.skew()

q1 = amounts.quantile(0.25)
q2 = amounts.quantile(0.50)
q3 = amounts.quantile(0.75)

iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

outliers_below = (amounts < lower_fence).sum()
outliers_above = (amounts > upper_fence).sum()

print("\n3. Completed order_amount statistics:")
print(f"Count: {amounts.count()}")
print(f"Mean: ₹{mean_amount:,.2f}")
print(f"Median: ₹{median_amount:,.2f}")
print(f"Skewness: {skewness:.2f}")
print(f"Q1 (25th percentile): ₹{q1:,.2f}")
print(f"Q2 (50th percentile / median): ₹{q2:,.2f}")
print(f"Q3 (75th percentile): ₹{q3:,.2f}")
print(f"IQR (Q3 - Q1): ₹{iqr:,.2f}")
print(f"Lower fence: ₹{lower_fence:,.2f}")
print(f"Upper fence: ₹{upper_fence:,.2f}")
print(f"Outliers below lower fence: {outliers_below}")
print(f"Outliers above upper fence: {outliers_above}")


3. Completed order_amount statistics:
Count: 11103
Mean: ₹269.86
Median: ₹135.00
Skewness: 4.62
Q1 (25th percentile): ₹64.80
Q2 (50th percentile / median): ₹135.00
Q3 (75th percentile): ₹297.00
IQR (Q3 - Q1): ₹232.20
Lower fence: ₹-283.50
Upper fence: ₹645.30
Outliers below lower fence: 0
Outliers above upper fence: 898


CORRELATION MATRIX

In [12]:
correlation_columns = [
    "order_amount", "discount_pct",
    "quantity", "delivery_minutes"
]

correlation_matrix = completed_orders[
    correlation_columns
].corr(method="pearson")

print("\n4. Correlation matrix for completed orders:")
print(correlation_matrix.round(3))

# Find the pair with the largest absolute correlation,
# excluding the diagonal (a variable correlated with itself).
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

correlation_pairs = upper_triangle.stack().dropna()

if not correlation_pairs.empty:
    strongest_pair = correlation_pairs.abs().idxmax()
    strongest_value = correlation_pairs.loc[strongest_pair]

    print(
        "\nLargest absolute correlation:",
        f"{strongest_pair[0]} and {strongest_pair[1]}",
        f"= {strongest_value:.3f}"
    )



4. Correlation matrix for completed orders:
                  order_amount  discount_pct  quantity  delivery_minutes
order_amount             1.000        -0.095     0.089            -0.003
discount_pct            -0.095         1.000    -0.001            -0.009
quantity                 0.089        -0.001     1.000            -0.015
delivery_minutes        -0.003        -0.009    -0.015             1.000

Largest absolute correlation: order_amount and discount_pct = -0.095
